In [ ]:
mesh_to_fit = inner_mesh_smooth_subset.copy()
# ==== Fit Sphere ====
sphere_params = analysis.fit_sphere(points=mesh_to_fit.vertices)
sphere_x0, sphere_y0, sphere_z0, sphere_radius = sphere_params
sphere_mesh = trimesh.creation.icosphere(radius=sphere_radius, subdivisions=8)
sphere_mesh.vertices += [sphere_x0, sphere_y0, sphere_z0]
datahandler.save_array(np.array(sphere_params)[:, np.newaxis].T, "sphere_fit", header="x,y,z,radius",
                       folderpath=resdata_dir)
# ==== Crop (Part of) Sphere ====
seg_fit_crop_cap_angle = 120.0
print(f">> Cropping sphere to {seg_fit_crop_cap_angle} degrees cap...")
sphere_crop_mask = ((sphere_mesh.vertices[:, 0] - sphere_x0) / np.linalg.norm(
    sphere_mesh.vertices - [sphere_x0, sphere_y0, sphere_z0], axis=1)) <= np.cos(
    np.radians(seg_fit_crop_cap_angle))
sphere_mesh_cropped = analysis.sel_submesh(mesh=sphere_mesh, mask=sphere_crop_mask)
print(f"Num of sphere vertices: {sphere_mesh_cropped.vertices.shape[0]}")

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_raw_sphere-fit.pdf"), cmap="Greens_r", show_mesh_normals=True,
                 meshes=[sphere_mesh_cropped], mesh_alpha=1.0)

In [ ]:
# ==== 3D Render Mesh(es) ====
visuals.view_mesh([sphere_mesh, sphere_mesh_cropped], mesh_opacities=[0.5, 1.0], mesh_colors=["white", "red"],
                  img=img_raw, scale=img_scale)

In [ ]:
sphere_x0, sphere_y0, sphere_z0, sphere_radius = datahandler.load_array("sphere_fit", folderpath=resdata_dir)[0, :]
print(f"sphere_x0 = {sphere_x0} {img_unit}")
print(f"sphere_y0 = {sphere_y0} {img_unit}")
print(f"sphere_z0 = {sphere_z0} {img_unit}")
print(f"sphere_radius = {sphere_radius} {img_unit}")
min_proj_dist = 0.0
max_proj_dist = 40.0
slice_proj = 0.5
num_proj_samples = int(np.max(img_scale) / np.min(img_scale))
proj_mode = "mean"
min_all = np.arange(min_proj_dist, max_proj_dist, slice_proj)
max_all = min_all + slice_proj
mask = max_all <= max_proj_dist
min_all, max_all = min_all[mask], max_all[mask]
proj_tasks = np.stack([
    min_all,
    max_all,
    np.full(min_all.shape, num_proj_samples)
], axis=1)

proj_radii = sphere_radius - (min_all + 0.5 * slice_proj)
print(f">> Projecting at radii {proj_radii} {img_unit} ...")
# Flip direction of projection for going "inwards"
proj_tasks[:, :2] *= -1

all_projections = np.empty((len(proj_tasks), len(sampl_mesh.vertices)))
for i, proj_task in enumerate(proj_tasks):
    dist_min, dist_max, dist_num = proj_task
    all_projections[i] = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, unit=img_unit, min_dist_per_vert=None,
                                            scale=img_scale, min_dist=dist_min, max_dist=dist_max, num_dist=dist_num,
                                            mode=proj_mode, show_proj=False, savefig="", normalise=False)
print("=======")
print(f"Projected {len(all_projections)} layers !")
proj_xcords, proj_ycords = analysis.spherical_project(pts=sampl_mesh.vertices,
                                                      ref_point=[sphere_x0, sphere_y0, sphere_z0],
                                                      rotate=[np.pi / 2, 0, np.pi])
radial_projection = analysis.create_radial_stack(values=all_projections,
                                                 phi_coords=proj_xcords,
                                                 theta_cords=proj_ycords,
                                                 grid_n=np.mean(img_dim[1:]), projection_radii=proj_radii)
radial_stack, stack_cords = radial_projection
mid_radial_stack_i = radial_stack.shape[0] // 2

visuals.plot_matrix(radial_stack[mid_radial_stack_i], figsize=(14, 8), origin="upper",
                    title=f"R = {proj_radii[mid_radial_stack_i]:,.2f} {img_unit}",
                    savefig=os.path.join(resfig_dir, f"radial-stack_{mid_radial_stack_i}.pdf"),
                    unit="px", colorbar=True, cmap="inferno")

np.savez_compressed(os.path.join(resdata_dir, "radial_projection.npz"), radial_stack=radial_stack,
                    stack_cords=stack_cords)
radial_stack = np.load(os.path.join(resdata_dir, "radial_projection.npz"))["radial_stack"]
stack_cords = np.load(os.path.join(resdata_dir, "radial_projection.npz"))["stack_cords"]

datahandler.save_tiff(radial_stack, filepath=os.path.join(resfig_dir, "radial-stack.tiff"), img_scale=img_scale, img_unit=img_unit)
visuals.view_colored_mesh_multiple([sampl_mesh, sampl_mesh], vert_colors_list=[
    visuals.color_scalar(all_projections[20], normalise=True, cmap="Greens_r"),
    visuals.color_scalar(all_projections[-1], normalise=True, cmap="Greens_r")])